<a href="https://www.kaggle.com/code/rellegabrinao/surecure-model?scriptVersionId=359939138" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# SureCure: product identification and out-of-scope guard

Kaggle notebook for the SureCure PJDSC 2026 entry. Runs top to bottom on a GPU session.

**Before you run**
1. Settings: Accelerator = GPU (T4 or P100), Internet = On (needed for the ImageNet weights and `onnxruntime`).
2. Add data: *Pharmaceutical Drugs and Vitamins Dataset V2* (vencerlanz09). The notebook finds the `Capsure Dataset` folder by itself.
3. Optional: add a dataset of out-of-scope medicine photos (for example CapCure Neozep and Alaxan) and set `OOD_DIR` in the config cell.

**What it produces** (in `/kaggle/working`)
- `metadata.csv`: one row per image (split, class, burst, capture time, size, hashes, role)
- `audit.json`, `lighting_baseline.json`, `metrics.json`, `history.csv`
- `figures/`: confusion matrix, capture-time plot, training curves
- `surecure_model/`: `surecure.onnx`, `labels.json`, `meta.json`, `best.pt`
- `surecure_outputs.zip`: everything above in one download

**Design rules this notebook enforces**
- Product ID is trained on the synthetic composites only. Real photos are used only for checkpoint selection (dev bursts), conformal calibration (dev bursts) and testing (test bursts).
- Real photos are split by capture burst, never by frame, because frames in one burst are near-duplicates.
- Each class was photographed in its own time window, so a lighting-only baseline is run to show how much lighting alone reveals the class.

## 1. Configuration

In [ ]:
import os, io, re, json, math, time, random, hashlib, shutil, zipfile
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageOps
import matplotlib
import matplotlib.pyplot as plt

SEED = 42
QUICK = False   # True = 5-minute smoke test (40 synthetic images per class, 1 epoch)
QUICK = os.environ.get("SC_QUICK", str(int(QUICK))) == "1"
EPOCHS = int(os.environ.get("SC_EPOCHS", "1" if QUICK else "8"))
BATCH = 128
LR = 3e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1
IMG = 224                  # model input size
CACHE = 256                # size of the preprocessed square images
DEV_BURSTS_PER_CLASS = 4   # real bursts per class held out for selection and calibration
ALPHA = 0.05               # target false-rejection rate of the out-of-scope guard
SYN_VAL_FRAC = 0.10
FEATS_PER_CLASS = 600      # synthetic images per class used to fit the Mahalanobis guard
OOD_DIR = os.environ.get("SC_OOD_DIR", "")   # optional folder of out-of-scope medicine photos
FILL = (124, 116, 104)     # ImageNet mean, used for letterbox padding
NUM_WORKERS = min(4, os.cpu_count() or 2)

OUT = Path(os.environ.get("SC_OUT", "/kaggle/working"))
CACHE_DIR = OUT / "cache"
FIG_DIR = OUT / "figures"
MODEL_DIR = OUT / "surecure_model"
for d in (CACHE_DIR, FIG_DIR, MODEL_DIR):
    d.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)

SPLIT_DIRS = {"Train Image": "train", "Val": "val", "Test": "test"}
LABEL_FIX = {"Tuseran": "Tuseran_Forte"}   # Train uses "Tuseran", Val/Test use "Tuseran_Forte"
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def find_root():
    env = os.environ.get("SC_DATA_ROOT")
    if env:
        return Path(env)
    # Walk /kaggle/input a few levels deep looking for "Capsure Dataset".
    base = Path("/kaggle/input")
    if base.exists():
        frontier = [base]
        for _ in range(5):
            nxt = []
            for p in frontier:
                try:
                    for c in p.iterdir():
                        if c.is_dir():
                            if c.name == "Capsure Dataset":
                                return c
                            nxt.append(c)
                except PermissionError:
                    pass
            frontier = nxt
    import kagglehub
    p = Path(kagglehub.dataset_download("vencerlanz09/pharmaceutical-drugs-and-vitamins-dataset-v2"))
    return next(x for x in p.rglob("Capsure Dataset") if x.is_dir())


ROOT = find_root()
print("Data root:", ROOT)
print("Quick mode:", QUICK, "| epochs:", EPOCHS, "| workers:", NUM_WORKERS)

## 2. Index, preprocess and hash every image

For each file this step:
- applies the EXIF rotation (the real photos are stored sideways with orientation flag 6);
- letterboxes it to a square with the ImageNet mean color and resizes it to 256 px, so real and synthetic images share one geometry;
- records the MD5 of the file, a 64-bit difference hash (real photos only) for near-duplicate detection, and simple lighting features (real photos only).

Burst ID = the `YYYYMMDD_HHMMSS` prefix of a real photo's file name. Frames in one burst are near-duplicates of one scene.

In [ ]:
NAME_RE = re.compile(r"(\d{8})_(\d{6})(?:_(\d+))?")   # searched anywhere in the name


def letterbox(img, fill=FILL):
    w, h = img.size
    s = max(w, h)
    canvas = Image.new("RGB", (s, s), fill)
    canvas.paste(img, ((s - w) // 2, (s - h) // 2))
    return canvas


def dhash(img):
    g = img.convert("L").resize((9, 8), Image.BILINEAR)
    a = np.asarray(g, dtype=np.int16)
    bits = (a[:, 1:] > a[:, :-1]).flatten()
    return "%016x" % int("".join("1" if b else "0" for b in bits), 2)


def lighting_features(img):
    a = np.asarray(img.convert("RGB").resize((64, 64), Image.BILINEAR), dtype=np.float32) / 255.0
    mean = a.reshape(-1, 3).mean(0)
    std = a.reshape(-1, 3).std(0)
    lum = 0.299 * a[..., 0] + 0.587 * a[..., 1] + 0.114 * a[..., 2]
    hist, _ = np.histogram(lum, bins=16, range=(0.0, 1.0), density=True)
    return np.concatenate([mean, std, hist / 16.0]).tolist()


def process(job):
    src, dst, is_real = job
    data = Path(src).read_bytes()
    out = {"md5": hashlib.md5(data).hexdigest()}
    img = Image.open(io.BytesIO(data))
    img = ImageOps.exif_transpose(img).convert("RGB")
    out["width"], out["height"] = img.size
    if is_real:
        out["dhash"] = dhash(img)
        out["light"] = lighting_features(img)
    if not Path(dst).exists():
        Path(dst).parent.mkdir(parents=True, exist_ok=True)
        letterbox(img).resize((CACHE, CACHE), Image.BILINEAR).save(dst, quality=95)
    return out


rows = []
for split_dir, split in SPLIT_DIRS.items():
    sdir = ROOT / split_dir
    if not sdir.exists():
        print("Missing split folder:", sdir)
        continue
    for cdir in sorted(p for p in sdir.iterdir() if p.is_dir()):
        cls = LABEL_FIX.get(cdir.name, cdir.name)
        files = sorted(p for p in cdir.iterdir() if p.suffix.lower() in IMG_EXT)
        if QUICK and split == "train":
            files = files[:40]
        for f in files:
            m = NAME_RE.search(f.stem)
            is_real = split != "train"
            if m:
                burst, frame = f"{m.group(1)}_{m.group(2)}", int(m.group(3) or 0)
                hhmm = f"{m.group(2)[:2]}:{m.group(2)[2:4]}"
            elif is_real:
                # No timestamp in the name: treat the file as its own one-frame burst.
                burst, frame, hhmm = f"nots_{cls}_{f.stem}", 0, None
            else:
                burst, frame, hhmm = None, None, None
            rows.append({
                "path": str(f),
                "split": split,
                "source": "synthetic" if split == "train" else "real",
                "cls_folder": cdir.name,
                "cls": cls,
                "burst": burst,
                "frame": frame,
                "capture_hhmm": hhmm,
                "cache": str(CACHE_DIR / split / cls / (f.stem + ".jpg")),
            })
meta = pd.DataFrame(rows)
print(meta.groupby(["split", "source"]).size())
nots = meta[(meta["source"] == "real") & meta["capture_hhmm"].isna()]
print(f"Real files without a timestamp in the name: {len(nots)}")
if len(nots):
    print(nots.groupby("cls").size().to_string())
    print("Examples:", [Path(p).name for p in nots["path"].head(10)])

t0 = time.time()
jobs = list(zip(meta["path"], meta["cache"], meta["source"] == "real"))
ctx = mp.get_context("fork")
with ProcessPoolExecutor(max_workers=NUM_WORKERS, mp_context=ctx) as ex:
    results = list(ex.map(process, jobs, chunksize=64))
res = pd.DataFrame(results)
meta = pd.concat([meta.reset_index(drop=True), res], axis=1)
print(f"Processed {len(meta)} images in {time.time() - t0:.0f} s")

## 3. Audit

Counts per split and class, exact duplicates across splits, bursts per class, and the capture-time window of each class.

In [ ]:
CLASSES = sorted(meta["cls"].unique())
C = len(CLASSES)
CLS2IDX = {c: i for i, c in enumerate(CLASSES)}
meta["y"] = meta["cls"].map(CLS2IDX)

counts = meta.pivot_table(index="cls", columns="split", values="path", aggfunc="count").fillna(0).astype(int)
print(counts)
print("\nTotals:", meta["split"].value_counts().to_dict())

# Exact duplicates (same bytes)
dup = meta.groupby("md5").agg(n=("path", "size"), splits=("split", lambda s: ",".join(sorted(set(s)))))
dup = dup[dup["n"] > 1]
cross = dup[dup["splits"].str.contains(",")]
print(f"\nExact duplicate groups: {len(dup)} | spanning more than one split: {len(cross)}")

real = meta[meta["source"] == "real"].copy()

# Bursts shared between Val and Test (frame-level split leakage)
bv = real.groupby(["cls", "split"])["burst"].apply(set).unstack()
shared = {c: len(bv.loc[c, "val"] & bv.loc[c, "test"]) for c in bv.index
          if isinstance(bv.loc[c, "val"], set) and isinstance(bv.loc[c, "test"], set)}
bursts_per_class = real.groupby("cls")["burst"].nunique()
print("\nBursts per class:", bursts_per_class.to_dict())
print("Bursts shared by Val and Test per class:", shared)

# Near-duplicate clusters inside each burst (Hamming distance <= 6 on the 64-bit dHash)
def hamming(a, b):
    return bin(int(a, 16) ^ int(b, 16)).count("1")

cluster_ids = {}
for burst, g in real.sort_values("frame").groupby("burst"):
    reps = []
    for idx, h in zip(g.index, g["dhash"]):
        for k, (rh, cid) in enumerate(reps):
            if hamming(h, rh) <= 6:
                cluster_ids[idx] = cid
                break
        else:
            cid = f"{burst}#{len(reps)}"
            reps.append((h, cid))
            cluster_ids[idx] = cid
real["dup_cluster"] = pd.Series(cluster_ids)
meta.loc[real.index, "dup_cluster"] = real["dup_cluster"]
print(f"Real frames: {len(real)} | near-duplicate clusters: {real['dup_cluster'].nunique()} | bursts: {real['burst'].nunique()}")

# Capture-time window per class (the lighting confound)
win = (real.dropna(subset=["capture_hhmm"]).groupby("cls")["capture_hhmm"]
       .agg(["min", "max"]).sort_values("min"))
print("\nCapture window per class:")
print(win.to_string())

fig, ax = plt.subplots(figsize=(8, 6))
for i, (c, r) in enumerate(win.iterrows()):
    t0_ = int(r["min"][:2]) * 60 + int(r["min"][3:])
    t1_ = int(r["max"][:2]) * 60 + int(r["max"][3:])
    ax.plot([t0_ / 60, max(t1_, t0_ + 1) / 60], [i, i], lw=6)
ax.set_yticks(range(len(win)))
ax.set_yticklabels(win.index)
ax.set_xlabel("Capture time (hour of day)")
ax.set_title("Real photos: each class was captured in its own time window")
fig.tight_layout()
fig.savefig(FIG_DIR / "capture_windows.png", dpi=150)
plt.show()

audit = {
    "counts": counts.to_dict(),
    "totals": meta["split"].value_counts().to_dict(),
    "exact_duplicate_groups": int(len(dup)),
    "exact_duplicates_across_splits": int(len(cross)),
    "bursts_per_class": bursts_per_class.to_dict(),
    "val_test_shared_bursts": shared,
    "real_frames": int(len(real)),
    "near_duplicate_clusters": int(real["dup_cluster"].nunique()),
    "capture_window": win.to_dict(orient="index"),
    "real_size_counts": {f"{w}x{h}": int(n) for (w, h), n in
                         real.groupby(["width", "height"]).size().sort_values(ascending=False).head(5).items()},
}
(OUT / "audit.json").write_text(json.dumps(audit, indent=2, default=str))

## 4. Splits

- **Synthetic** (Train Image folder): 90% train, 10% validation (random).
- **Real** (Val and Test folders merged): per class, `DEV_BURSTS_PER_CLASS` bursts become **dev** (checkpoint choice and conformal calibration), the rest become **test**. A burst never appears in two roles.
- **Calibration set**: one random frame per dev burst, pooled across classes (n = bursts, not frames, so the conformal guarantee is not inflated by near-duplicates).

In [ ]:
rng = np.random.default_rng(SEED)
meta["role"] = None

syn_idx = np.array(meta.index[meta["source"] == "synthetic"], copy=True)
rng.shuffle(syn_idx)
n_val = int(len(syn_idx) * SYN_VAL_FRAC)
meta.loc[syn_idx[:n_val], "role"] = "syn_val"
meta.loc[syn_idx[n_val:], "role"] = "syn_train"

meta["is_cal"] = False
for c in CLASSES:
    bursts = sorted(meta.loc[(meta["source"] == "real") & (meta["cls"] == c), "burst"].unique())
    rng.shuffle(bursts)
    k = min(DEV_BURSTS_PER_CLASS, max(1, len(bursts) // 3))
    dev, test = set(bursts[:k]), set(bursts[k:])
    real_c = (meta["source"] == "real") & (meta["cls"] == c)
    meta.loc[real_c & meta["burst"].isin(dev), "role"] = "real_dev"
    meta.loc[real_c & meta["burst"].isin(test), "role"] = "real_test"
    for b in dev:
        idx = meta.index[real_c & (meta["burst"] == b)].to_numpy()
        meta.loc[rng.choice(idx), "is_cal"] = True

print(meta["role"].value_counts())
print("Calibration frames (one per dev burst):", int(meta["is_cal"].sum()))
assert meta["role"].notna().all()
assert not (set(meta.loc[meta.role == "real_dev", "burst"]) & set(meta.loc[meta.role == "real_test", "burst"]))
meta.drop(columns=["light"]).to_csv(OUT / "metadata.csv", index=False)

## 5. Lighting-only baseline

A logistic regression on 22 lighting numbers per real photo (mean and spread of R, G, B and a 16-bin brightness histogram), cross-validated by burst. Chance is 1/20 = 5%. A score far above chance means lighting alone identifies the class in the real photos, so any model fine-tuned on these photos could learn lighting instead of packaging. This is why product ID is trained on synthetic images only.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GroupKFold, cross_val_predict


def wilson(k, n, z=1.96):
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n
    den = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / den
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return (centre - half, centre + half)


R = meta[meta["source"] == "real"]
X = np.array(R["light"].tolist(), dtype=np.float32)
y = R["y"].to_numpy()
groups = R["burst"].to_numpy()
n_splits = min(5, len(np.unique(groups)))
clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000))
pred = cross_val_predict(clf, X, y, groups=groups, cv=GroupKFold(n_splits=n_splits))
acc = float((pred == y).mean())

# Burst-level: majority vote of frames in each burst
bdf = pd.DataFrame({"burst": groups, "y": y, "p": pred})
bvote = bdf.groupby("burst").agg(y=("y", "first"), p=("p", lambda s: s.mode().iloc[0]))
bk, bn = int((bvote["y"] == bvote["p"]).sum()), len(bvote)
lighting = {
    "frame_accuracy": acc,
    "burst_accuracy": bk / bn,
    "burst_accuracy_wilson95": wilson(bk, bn),
    "n_bursts": bn,
    "chance": 1 / C,
}
print(json.dumps(lighting, indent=2))
(OUT / "lighting_baseline.json").write_text(json.dumps(lighting, indent=2))

## 6. Product identification: ResNet-18 trained on synthetic images

Augmentations push synthetic images toward the look of the real phone photos: random letterbox bars, crops, perspective, color shifts, blur, down-and-up scaling (the real photos are only 300 × 135 px) and JPEG artifacts. No horizontal flip, because mirrored packaging text does not occur in real photos.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import transforms as T

torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True
device = "cuda" if torch.cuda.is_available() else "cpu"
use_amp = device == "cuda"
print("Device:", device, "| torch", torch.__version__, "| torchvision", torchvision.__version__)

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]


class RandomLetterbox:
    # Crop a random narrow vertical strip and pad it back to a square, like the portrait real photos.
    def __init__(self, p=0.5, min_aspect=0.4):
        self.p, self.min_aspect = p, min_aspect

    def __call__(self, img):
        if random.random() > self.p:
            return img
        w, h = img.size
        a = random.uniform(self.min_aspect, 1.0)
        nw = max(8, int(w * a))
        x0 = random.randint(0, w - nw)
        return letterbox(img.crop((x0, 0, x0 + nw, h))).resize((w, h), Image.BILINEAR)


class RandomDownUp:
    def __init__(self, p=0.5, lo=80, hi=160):
        self.p, self.lo, self.hi = p, lo, hi

    def __call__(self, img):
        if random.random() > self.p:
            return img
        s = random.randint(self.lo, self.hi)
        return img.resize((s, s), Image.BILINEAR).resize(img.size, Image.BILINEAR)


class RandomJPEG:
    def __init__(self, p=0.5, lo=30, hi=90):
        self.p, self.lo, self.hi = p, lo, hi

    def __call__(self, img):
        if random.random() > self.p:
            return img
        buf = io.BytesIO()
        img.save(buf, format="JPEG", quality=random.randint(self.lo, self.hi))
        buf.seek(0)
        return Image.open(buf).convert("RGB")


train_tf = T.Compose([
    RandomLetterbox(0.5),
    T.RandomResizedCrop(IMG, scale=(0.5, 1.0), ratio=(0.75, 1.33)),
    T.RandomApply([T.RandomPerspective(distortion_scale=0.2, p=1.0)], p=0.3),
    T.ColorJitter(0.3, 0.3, 0.3, 0.05),
    T.RandomApply([T.GaussianBlur(5, sigma=(0.1, 2.0))], p=0.3),
    RandomDownUp(0.5),
    RandomJPEG(0.5),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])
eval_tf = T.Compose([T.Resize((IMG, IMG)), T.ToTensor(), T.Normalize(MEAN, STD)])


class ImgDS(Dataset):
    def __init__(self, df, tf):
        self.paths = df["cache"].tolist()
        self.ys = df["y"].astype(int).tolist()
        self.tf = tf

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        return self.tf(Image.open(self.paths[i]).convert("RGB")), self.ys[i]


def loader(df, tf, shuffle, bs=BATCH):
    return DataLoader(ImgDS(df, tf), batch_size=bs, shuffle=shuffle, num_workers=NUM_WORKERS,
                      pin_memory=(device == "cuda"), persistent_workers=NUM_WORKERS > 0,
                      drop_last=shuffle)


syn_train = meta[meta.role == "syn_train"]
syn_val = meta[meta.role == "syn_val"]
real_dev = meta[meta.role == "real_dev"]
real_test = meta[meta.role == "real_test"]
dl_train = loader(syn_train, train_tf, True)
dl_synval = loader(syn_val, eval_tf, False)
dl_dev = loader(real_dev, eval_tf, False)
print(len(syn_train), "train |", len(syn_val), "syn val |", len(real_dev), "real dev |", len(real_test), "real test")

In [ ]:
model = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, C)
model = model.to(device)

opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
steps = EPOCHS * len(dl_train)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=steps, pct_start=0.1)
crit = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
try:
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
except (AttributeError, TypeError):
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)


@torch.no_grad()
def predict(m, dl):
    m.eval()
    out = []
    for x, _ in dl:
        with torch.autocast(device_type=device, dtype=torch.float16, enabled=use_amp):
            out.append(m(x.to(device, non_blocking=True)).float().cpu())
    return torch.cat(out)


history, best = [], (-1.0, None)
for ep in range(EPOCHS):
    model.train()
    t0, tot, n = time.time(), 0.0, 0
    for x, yb in dl_train:
        x, yb = x.to(device, non_blocking=True), yb.to(device, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device, dtype=torch.float16, enabled=use_amp):
            loss = crit(model(x), yb)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        sched.step()
        tot += loss.item() * len(yb)
        n += len(yb)
    syn_acc = float((predict(model, dl_synval).argmax(1).numpy() == syn_val["y"].to_numpy()).mean())
    dev_acc = float((predict(model, dl_dev).argmax(1).numpy() == real_dev["y"].to_numpy()).mean())
    history.append({"epoch": ep + 1, "train_loss": tot / max(n, 1), "syn_val_acc": syn_acc,
                    "real_dev_acc": dev_acc, "seconds": time.time() - t0})
    print(history[-1])
    if dev_acc > best[0]:
        best = (dev_acc, ep + 1)
        torch.save(model.state_dict(), MODEL_DIR / "best.pt")

pd.DataFrame(history).to_csv(OUT / "history.csv", index=False)
model.load_state_dict(torch.load(MODEL_DIR / "best.pt", map_location=device))
print("Best epoch by real-dev accuracy:", best[1], "| dev acc:", round(best[0], 4))

h = pd.DataFrame(history)
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(h["epoch"], h["syn_val_acc"], marker="o", label="synthetic validation")
ax.plot(h["epoch"], h["real_dev_acc"], marker="o", label="real dev bursts")
ax.set_xlabel("Epoch"); ax.set_ylabel("Top-1 accuracy"); ax.set_ylim(0, 1); ax.legend()
ax.set_title("Synthetic-to-real gap during training")
fig.tight_layout(); fig.savefig(FIG_DIR / "training_curves.png", dpi=150); plt.show()

## 7. Test on held-out real bursts

Reported three ways: per frame, per near-duplicate cluster (one frame per cluster), and per burst (majority vote). The burst level is the honest sample size for confidence intervals.

In [ ]:
from sklearn.metrics import f1_score, confusion_matrix

dl_test = loader(real_test, eval_tf, False)
logits_test = predict(model, dl_test)
p_test = logits_test.argmax(1).numpy()
y_test = real_test["y"].to_numpy()

frame_acc = float((p_test == y_test).mean())
macro_f1 = float(f1_score(y_test, p_test, average="macro"))
T_ = real_test.assign(p=p_test)
clus = T_.groupby("dup_cluster").first()
clus_k, clus_n = int((clus["p"] == clus["y"]).sum()), len(clus)
bv_ = T_.groupby("burst").agg(y=("y", "first"), p=("p", lambda s: s.mode().iloc[0]))
bk_, bn_ = int((bv_["p"] == bv_["y"]).sum()), len(bv_)
per_class = T_.groupby("cls").apply(lambda g: float((g["p"] == g["y"]).mean())).to_dict()

syn_val_acc = float((predict(model, dl_synval).argmax(1).numpy() == syn_val["y"].to_numpy()).mean())
metrics = {
    "best_epoch": best[1],
    "synthetic_val_accuracy": syn_val_acc,
    "real_test_frame_accuracy": frame_acc,
    "real_test_macro_f1": macro_f1,
    "real_test_cluster_accuracy": clus_k / clus_n, "n_clusters": clus_n,
    "real_test_burst_accuracy": bk_ / bn_, "n_bursts": bn_,
    "real_test_burst_accuracy_wilson95": wilson(bk_, bn_),
    "per_class_frame_accuracy": per_class,
    "lighting_only_baseline_burst_accuracy": lighting["burst_accuracy"],
}
print(json.dumps({k: v for k, v in metrics.items() if k != "per_class_frame_accuracy"}, indent=2))

cm = confusion_matrix(y_test, p_test, labels=range(C), normalize="true")
fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(cm, vmin=0, vmax=1, cmap="Blues")
ax.set_xticks(range(C)); ax.set_xticklabels(CLASSES, rotation=90)
ax.set_yticks(range(C)); ax.set_yticklabels(CLASSES)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title("Real test bursts: row-normalized confusion matrix")
fig.colorbar(im, ax=ax, fraction=0.046)
fig.tight_layout(); fig.savefig(FIG_DIR / "confusion_matrix.png", dpi=150); plt.show()

In [ ]:
# Show real test photos wrongly predicted as Myra_E
TARGET = "Myra_E"
t_idx = CLASSES.index(TARGET)
wrong = T_[(T_["p"] == t_idx) & (T_["y"] != t_idx)]
print(f"{len(wrong)} test frames wrongly predicted as {TARGET}")
print(wrong["cls"].value_counts().to_string())        # which true classes leak into it

# One frame per burst, so you don't see 10 copies of the same shot
sample = wrong.groupby("burst").head(1).sample(min(20, wrong["burst"].nunique()), random_state=0)

fig, axes = plt.subplots(4, 5, figsize=(15, 12))
for ax, (_, r) in zip(axes.flat, sample.iterrows()):
    ax.imshow(ImageOps.exif_transpose(Image.open(r["path"])))   # original photo, upright
    ax.set_title(f"true: {r['cls']}", fontsize=9)
    ax.axis("off")
for ax in axes.flat[len(sample):]:
    ax.axis("off")
fig.suptitle(f"Real test photos predicted as {TARGET}")
fig.tight_layout()
fig.savefig(FIG_DIR / f"false_{TARGET}.png", dpi=120)
plt.show()

# For comparison: genuine Myra_E, real and synthetic
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
real_myra = T_[T_["y"] == t_idx].groupby("burst").head(1).head(5)
syn_myra = meta[(meta["role"] == "syn_train") & (meta["cls"] == TARGET)].sample(5, random_state=0)
for ax, p in zip(axes[0], real_myra["path"]):
    ax.imshow(ImageOps.exif_transpose(Image.open(p))); ax.set_title("real Myra_E"); ax.axis("off")
for ax, p in zip(axes[1], syn_myra["path"]):
    ax.imshow(Image.open(p)); ax.set_title("synthetic Myra_E"); ax.axis("off")
fig.tight_layout(); plt.show()

## 8. Out-of-scope guard: Mahalanobis score with a conformal threshold

1. Take 512-d penultimate features of synthetic training images; fit one mean per class and one shared covariance (Lee et al., 2018).
2. Score an image by its smallest Mahalanobis distance to any class mean, then standardize it with that class's mean and spread on synthetic validation images.
3. Threshold: with n calibration scores (one frame per dev burst), take the k-th smallest, k = ⌈(n+1)(1−α)⌉. This bounds the rate of wrongly rejected in-scope photos at α for photos taken like the calibration set (Bates et al., 2023). It needs n ≥ 1/α − 1 = 19.

In [ ]:
backbone = nn.Sequential(*list(model.children())[:-1], nn.Flatten(1)).eval()


@torch.no_grad()
def features(df):
    out = []
    for x, _ in loader(df, eval_tf, False):
        out.append(backbone(x.to(device)).float().cpu())
    return torch.cat(out).numpy().astype(np.float64)


fit_df = syn_train.groupby("cls", group_keys=False).apply(
    lambda g: g.sample(min(len(g), FEATS_PER_CLASS), random_state=SEED))
F_fit = features(fit_df)
y_fit = fit_df["y"].to_numpy()
mu = np.stack([F_fit[y_fit == c].mean(0) for c in range(C)])
centered = F_fit - mu[y_fit]
cov = centered.T @ centered / len(F_fit)
cov += np.eye(cov.shape[0]) * 1e-6 * np.trace(cov) / cov.shape[0]
prec = np.linalg.inv(cov)


def maha(Fm):
    d = np.stack([np.einsum("nd,de,ne->n", Fm - mu[c], prec, Fm - mu[c]) for c in range(C)], 1)
    return d.min(1), d.argmin(1)


s_val, c_val = maha(features(syn_val))
glob_m, glob_s = s_val.mean(), s_val.std() + 1e-9
m_c = np.array([s_val[c_val == c].mean() if (c_val == c).sum() >= 10 else glob_m for c in range(C)])
sd_c = np.array([s_val[c_val == c].std() + 1e-9 if (c_val == c).sum() >= 10 else glob_s for c in range(C)])


def zscore(Fm):
    s, c = maha(Fm)
    return (s - m_c[c]) / sd_c[c]


cal = meta[meta.is_cal]
z_cal = np.sort(zscore(features(cal)))
n_cal = len(z_cal)
k = math.ceil((n_cal + 1) * (1 - ALPHA))
if k > n_cal:
    print(f"WARNING: n_cal={n_cal} is too small for alpha={ALPHA}; threshold set to the max score.")
    k = n_cal
thr = float(z_cal[k - 1])

z_test = zscore(features(real_test))
rej = z_test > thr
T_ = T_.assign(z=z_test, rejected=rej)
one_per_burst = T_.groupby("burst").sample(1, random_state=SEED)
fr_k, fr_n = int(one_per_burst["rejected"].sum()), len(one_per_burst)
guard = {
    "alpha": ALPHA, "n_cal": n_cal, "k": k, "threshold_z": thr,
    "false_rejection_all_test_frames": float(rej.mean()),
    "false_rejection_one_frame_per_burst": fr_k / fr_n, "n_bursts": fr_n,
    "false_rejection_one_frame_per_burst_wilson95": wilson(fr_k, fr_n),
}

ood_files = [p for p in Path(OOD_DIR).rglob("*") if p.suffix.lower() in IMG_EXT] if OOD_DIR and Path(OOD_DIR).exists() else []
if ood_files:
    from sklearn.metrics import roc_auc_score
    ood_cache = []
    for i, p in enumerate(ood_files):
        dst = CACHE_DIR / "ood" / f"{i:06d}.jpg"
        process((str(p), str(dst), False))
        ood_cache.append(str(dst))
    ood_df = pd.DataFrame({"cache": ood_cache, "y": 0})
    z_ood = zscore(features(ood_df))
    labels = np.r_[np.zeros(len(one_per_burst)), np.ones(len(z_ood))]
    scores = np.r_[one_per_burst["z"].to_numpy(), z_ood]
    guard.update({"ood_n": len(z_ood), "ood_auroc_vs_one_frame_per_test_burst": float(roc_auc_score(labels, scores)),
                  "ood_detected_at_threshold": float((z_ood > thr).mean())})
else:
    print("No OOD_DIR given: AUROC against out-of-scope medicines is skipped.")
metrics["ood_guard"] = guard
print(json.dumps(guard, indent=2))
(OUT / "metrics.json").write_text(json.dumps(metrics, indent=2, default=float))

## 9. Export to ONNX for the Android app

One file, two outputs:
- `logits` (1 × 20): product scores, order given in `labels.json`;
- `ood_z` (1): standardized out-of-scope score; reject the product ID when `ood_z > threshold_z` from `meta.json`.

Input: `image`, float32, shape 1 × 3 × 224 × 224, RGB scaled to 0..1, after EXIF rotation and letterboxing to a square with the fill color in `meta.json`. ImageNet normalization happens inside the model.

In [ ]:
class SureCureNet(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.backbone = nn.Sequential(*list(model.children())[:-1], nn.Flatten(1))
        self.fc = model.fc
        self.register_buffer("img_mean", torch.tensor(MEAN).view(1, 3, 1, 1))
        self.register_buffer("img_std", torch.tensor(STD).view(1, 3, 1, 1))
        self.register_buffer("mu", torch.tensor(mu, dtype=torch.float32))
        self.register_buffer("prec", torch.tensor(prec, dtype=torch.float32))
        self.register_buffer("m_c", torch.tensor(m_c, dtype=torch.float32))
        self.register_buffer("sd_c", torch.tensor(sd_c, dtype=torch.float32))

    def forward(self, x):
        f = self.backbone((x - self.img_mean) / self.img_std)
        logits = self.fc(f)
        diff = f.unsqueeze(1) - self.mu.unsqueeze(0)            # B x C x D
        d = (torch.matmul(diff, self.prec) * diff).sum(-1)      # B x C
        s, c = torch.min(d, dim=1)
        z = (s - self.m_c[c]) / self.sd_c[c]
        return logits, z


net = SureCureNet(model).float().cpu().eval()
dummy = torch.rand(1, 3, IMG, IMG)
onnx_path = MODEL_DIR / "surecure.onnx"
export_kwargs = dict(input_names=["image"], output_names=["logits", "ood_z"],
                     dynamic_axes={"image": {0: "batch"}, "logits": {0: "batch"}, "ood_z": {0: "batch"}},
                     opset_version=17)
import inspect
if "dynamo" in inspect.signature(torch.onnx.export).parameters:
    export_kwargs["dynamo"] = False   # classic exporter: stable for this graph
torch.onnx.export(net, dummy, str(onnx_path), **export_kwargs)
print("Saved", onnx_path, f"{onnx_path.stat().st_size / 1e6:.1f} MB")

(MODEL_DIR / "labels.json").write_text(json.dumps(CLASSES, indent=2))
(MODEL_DIR / "meta.json").write_text(json.dumps({
    "input": {"name": "image", "shape": [1, 3, IMG, IMG], "dtype": "float32", "range": [0, 1], "channels": "RGB",
              "preprocess": "EXIF rotate upright, letterbox to square with fill_rgb, resize to 224x224 bilinear, divide by 255"},
    "fill_rgb": list(FILL),
    "outputs": {"logits": "product scores in labels.json order", "ood_z": "reject product ID if ood_z > threshold_z"},
    "threshold_z": thr, "alpha": ALPHA, "n_cal": n_cal,
    "best_epoch": best[1], "seed": SEED,
}, indent=2))

In [ ]:
# Parity check: ONNX Runtime must reproduce PyTorch outputs.
try:
    import onnxruntime as ort
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnxruntime"], check=True)
    import onnxruntime as ort

sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
sample = real_test.sample(min(64, len(real_test)), random_state=SEED)
x = torch.stack([T.Compose([T.Resize((IMG, IMG)), T.ToTensor()])(Image.open(p).convert("RGB")) for p in sample["cache"]])
with torch.no_grad():
    lt, zt = net(x)
lo, zo = sess.run(None, {"image": x.numpy()})
dl, dz = float(np.abs(lo - lt.numpy()).max()), float(np.abs(zo - zt.numpy()).max())
print(f"max |logits diff| = {dl:.2e}, max |ood_z diff| = {dz:.2e}")
agree = float((lo.argmax(1) == lt.numpy().argmax(1)).mean())
print("Top-1 agreement:", agree)
assert agree >= 0.98 and dl < 1e-2, "ONNX and PyTorch outputs differ more than expected"

## 10. Package outputs

In [ ]:
zip_path = OUT / "surecure_outputs.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in [OUT / "metadata.csv", OUT / "audit.json", OUT / "lighting_baseline.json",
              OUT / "metrics.json", OUT / "history.csv"]:
        if p.exists():
            z.write(p, p.name)
    for d in (FIG_DIR, MODEL_DIR):
        for p in d.glob("*"):
            z.write(p, f"{d.name}/{p.name}")
print("Download:", zip_path, f"{zip_path.stat().st_size / 1e6:.1f} MB")
shutil.rmtree(CACHE_DIR, ignore_errors=True)   # keep /kaggle/working small